In [ ]:
import numpy as np
import xarray as xr

# --- Observed Wind Speed ---
u_true = eval_targets["10m_u_component_of_wind"].isel(time=0).squeeze("batch")
v_true = eval_targets["10m_v_component_of_wind"].isel(time=0).squeeze("batch")
wind_true = np.sqrt(u_true**2 + v_true**2)
region_true = wind_true.sel(lat=slice(54, 53), lon=slice(9.5, 10.5))
mean_true = region_true.mean().item()

# --- Predicted Wind Speed ---
u_pred = predictions["10m_u_component_of_wind"].isel(time=0).squeeze("batch")
v_pred = predictions["10m_v_component_of_wind"].isel(time=0).squeeze("batch")
wind_pred = np.sqrt(u_pred**2 + v_pred**2)
region_pred = wind_pred.sel(lat=slice(54, 53), lon=slice(9.5, 10.5))
mean_pred = region_pred.mean().item()

# --- Print Results ---
print(f"✅ Mean observed 10m wind speed:   {mean_true:.2f} m/s")
print(f"✅ Mean predicted 10m wind speed: {mean_pred:.2f} m/s")
print(f"🔎 Absolute difference:           {abs(mean_pred - mean_true):.2f} m/s")


### For 0.25 resolution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import xarray as xr
import numpy as np

# 1. Compute wind speed
u_true = eval_targets["10m_u_component_of_wind"].isel(time=0).squeeze("batch")
v_true = eval_targets["10m_v_component_of_wind"].isel(time=0).squeeze("batch")
wind_true = (u_true**2 + v_true**2) ** 0.5

u_pred = predictions["10m_u_component_of_wind"].isel(time=0).squeeze("batch")
v_pred = predictions["10m_v_component_of_wind"].isel(time=0).squeeze("batch")
wind_pred = (u_pred**2 + v_pred**2) ** 0.5

# 2. Select region (Hamburg area)
region = dict(lat=slice(54.0, 53.0), lon=slice(9.5, 10.5))  # south to north
true_region = wind_true.sel(**region)
pred_region = wind_pred.sel(**region)

# 3. Load Hamburg boundary
gdf = gpd.read_file("data/gadm41_DEU_2.json")  # Adjust if needed
hamburg_shape = gdf[gdf["NAME_1"] == "Hamburg"]

# 4. Plotting
fig, axs = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"projection": ccrs.PlateCarree()})

for ax, data, title in zip(
    axs,
    [true_region, pred_region],
    ["Observed Wind Speed at 21:00 UTC", "Predicted Wind Speed at 21:00 UTC"]
):
    ax.set_extent([9.5, 10.5, 53.0, 54.0], crs=ccrs.PlateCarree())
    data.plot.pcolormesh(ax=ax, cmap="viridis", add_colorbar=True, transform=ccrs.PlateCarree())
    
    # Map features
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
    ax.add_feature(cfeature.RIVERS, edgecolor='blue', linewidth=0.3)
    
    # Hamburg border and label
    hamburg_shape.boundary.plot(ax=ax, edgecolor="black", linewidth=1.2, transform=ccrs.PlateCarree())
    ax.plot(10.0, 53.55, 'ro', markersize=5, transform=ccrs.PlateCarree())
    ax.text(10.02, 53.55, "Hamburg", color='red', transform=ccrs.PlateCarree())
    
    # Gridlines
    gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}
    gl.xlocator = mticker.MultipleLocator(0.25)
    gl.ylocator = mticker.MultipleLocator(0.25)

    ax.set_title(title)

plt.tight_layout()
plt.show()


### For 1.0 resolution

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib.ticker as mticker

# Load Hamburg boundary
hamburg_shape = gpd.read_file("data/gadm41_DEU_2.json")
hamburg_shape = hamburg_shape[hamburg_shape["NAME_1"] == "Hamburg"]

# Compute wind speed from components
u_true = eval_targets["10m_u_component_of_wind"].isel(time=0).squeeze("batch")
v_true = eval_targets["10m_v_component_of_wind"].isel(time=0).squeeze("batch")
wind_true = np.sqrt(u_true**2 + v_true**2)

u_pred = predictions["10m_u_component_of_wind"].isel(time=0).squeeze("batch")
v_pred = predictions["10m_v_component_of_wind"].isel(time=0).squeeze("batch")
wind_pred = np.sqrt(u_pred**2 + v_pred**2)

# Define region
lon_min, lon_max = 8.5, 11.5
lat_min, lat_max = 52.5, 54.5
region = dict(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max))
true_region = wind_true.sel(**region)
pred_region = wind_pred.sel(**region)

# Shared color scale limits
vmin = min(true_region.min().item(), pred_region.min().item())
vmax = max(true_region.max().item(), pred_region.max().item())

# Plot
fig, axs = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': ccrs.PlateCarree()})

for ax, data, title in zip(axs, [true_region, pred_region], ["Observed Wind Speed at 21:00 UTC", "Predicted Wind Speed at 21:00 UTC"]):
    im = data.plot.pcolormesh(
        ax=ax, cmap="viridis", add_colorbar=False, transform=ccrs.PlateCarree(),
        vmin=vmin, vmax=vmax
    )
    ax.set_title(title)
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4)
    ax.add_feature(cfeature.RIVERS, edgecolor='blue', linewidth=0.3)
    hamburg_shape.boundary.plot(ax=ax, edgecolor="black", linewidth=1.2, transform=ccrs.PlateCarree())
    ax.plot(10.0, 53.55, 'ro', markersize=6, transform=ccrs.PlateCarree())
    ax.text(10.02, 53.55, "Hamburg", color='red', transform=ccrs.PlateCarree())
   # gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.7, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}
    gl.xlocator = mticker.MultipleLocator(0.5)
    gl.ylocator = mticker.MultipleLocator(0.5)

# Add shared colorbar
cbar = fig.colorbar(im, ax=axs, orientation='vertical', fraction=0.025, pad=0.02)
cbar.set_label("10m Wind Speed (m/s)")

plt.subplots_adjust(wspace=0.15, right=0.87)
plt.show()